# Bagging

This notebook implements a **Bagging (Bootstrap Aggregating)** approach to fake news detection, following the optimization problem defined in the project specification. The objective minimises a weighted, category-balanced cross-entropy loss with regularisation.

# Optimization Problem

MIN Z

$$
\min_{\theta_{1}, \dots, \theta_{M}} \quad Z^{(e)} = - \frac{1}{N} \sum_{i=1}^{N} \alpha_{c_i} [w_1 y_i \log(F^{(e)}(x)) + w_0 (1 - y_i) \log(1 - F^{(e)}(x))] + \sum_{m=1}^{M} \lambda_{m} \Omega(\theta_{m})
$$

Where

- $e \in \{bag, boost, stack\}$
- $F(x) \in \{0, 1\}$

<br>SUBJECT TO <br><br>

$C_{1}^{bag}$: Ensemble Prediction Function

$$F(x) = \frac{1}{M} \displaystyle\sum_{m=1}^{M} f_m(x)$$

Where

- $f_m$ is trained on a bootstrap sample $B_m \subset \mathcal{D}$ with replacement

$C_2$: Feature Mapping

$$x_i = \phi (title_i, text_i, category_i, dataset_i)$$

$C_3$: Label Constraint

$$y_i \in \{0, 1\}, \qquad 0 = fake, \quad 1 = real$$

$C_4$: Category Weight

$$\alpha_{c_i} = \frac{N}{K \cdot N_{c_i}}$$

Where

- $K$ is the number of distinct categories
- $N_{c_i}$ is the number of samples in category $c_i$
- $N$ is the total number of samples

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$

Where

- $N_1$: number of real samples
- $N_0$: number of fake samples
- $N = N_0 + N_1$


# Environment Configuration

The following code cell contains the cudf pandas magic command. This command is kept seperate so cudf.pandas does not try to reload when adding new imports.

In [1]:
%load_ext cudf.pandas

The following code cell contains the dependencies that will be used in this notebook.

In [6]:
import re
import warnings
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

warnings.filterwarnings('ignore')
np.random.seed(42)
cp.random.seed(42)

# Colab Configuration

Run the code cell below to download the data ingestion script from Github.

In [3]:
!wget https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py

--2026-05-08 02:39:18--  https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8393 (8.2K) [text/plain]
Saving to: ‘ingest_data.py.1’

ingest_data.py.1    100%[===================>]   8.20K  --.-KB/s    in 0s      

2026-05-08 02:39:18 (45.4 MB/s) - ‘ingest_data.py.1’ saved [8393/8393]



# Data Ingestion

Load the unified fake-news DataFrame using the shared `ingest_data` script. This combines three Kaggle datasets (`bhavikjikadara`, `mahdimashayekhi`, `shawkyelgendy`) and returns a DataFrame with columns: `title`, `text`, `label`, `category`, `dataset`. Basic preprocessing (null-text removal, deduplication, category normalisation) is applied inside the loader.

In [4]:
from ingest_data import load_datasets
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...


[100593][02:39:21:342577][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[bhavik] Loaded 'true.csv': 21417 rows


[100593][02:39:23:750716][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[bhavik] Loaded 'fake.csv': 23481 rows

Loading mahdimashayekhi ...


...

[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...


[100593][02:39:25:319586][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[shawky] Loaded 'real.csv': 21871 rows


[100593][02:39:25:585378][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[shawky] Loaded 'fake.csv': 20072 rows

Dropped 649 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,542
Fake (0): 47,161
Real (1): 52,381

Rows per source:
shawkyelgendy          40,898
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,765
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Technology             2,882
  Business               2,849
  Science                2,789
------------------------------------------------------------


# Text Preprocessing

In [8]:
STOP_WORDS = ENGLISH_STOP_WORDS

def clean_text(s: str) -> str:
    """Normalise a raw text string for TF-IDF vectorization."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'&[a-z]+;', ' ', s)
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [tok for tok in s.split() if tok not in STOP_WORDS]
    return ' '.join(tokens)

# cudf.pandas accelerates these apply calls on GPU where UDF is supported
df['clean_title'] = df['title'].apply(clean_text)
df['clean_text']  = df['text'].apply(clean_text)

df['combined_text'] = df['clean_title'] + ' ' + df['clean_text']

print(f"Rows after preprocessing : {len(df):,}")
print(f"\nSample combined_text:")
print(df['combined_text'].iloc[0][:300])

Rows after preprocessing : 99,542

Sample combined_text:
budget fight looms republicans flip fiscal script washington reuters head conservative republican faction congress voted month huge expansion national debt pay tax cuts called fiscal conservative sunday urged budget restraint keeping sharp pivot way republicans representative mark meadows speaking c


___